In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 4.5 The Polar Decomposition and Procrustes

In [ ]:
from ecp.style import header, use_style

use_style()
header(
    volume="Volume IV — The Singular Value Decomposition",
    number="4.5",
    title="The Polar Decomposition and Procrustes",
    blurb="Every matrix is a rotation times a stretch, and the rotation is "
    "the nearest orthogonal matrix there is — ten thousand challengers lose. "
    "The same factor solves Procrustes: given two point clouds, the "
    "best-aligning rotation is one SVD away, with a determinant correction "
    "crystallographers wrote down in 1976.",
    difficulty="advanced",
    estimate="90–120 min",
)

## Notebook overview

[§4.1](svd-geometry.ipynb) read the SVD as rotate–stretch–rotate. This
notebook regroups the same three factors into two: every square matrix is
**one rotation times one symmetric stretch**, $A = QH$ — the matrix version
of writing a complex number as $re^{i\theta}$. The factor $Q$ is not merely
*a* rotation associated with $A$; it is the **nearest orthogonal matrix to
$A$** in the Frobenius norm, and we certify that the way
[§4.2](low-rank-eckart-young.ipynb) certified Eckart–Young: $10^{4}$
random orthogonal challengers, every one of which loses by a measured
margin.

Nearness to a rotation is not an ornament — it is a problem industries
solve daily. Given two point clouds, one a rotated (possibly scaled,
shifted, noisy) copy of the other, the **orthogonal Procrustes problem**
asks for the rotation that best aligns them, and its exact solution is the
polar factor of a $2\times2$ correlation matrix: one SVD, no iteration.
Crystallography, protein-structure comparison and shape statistics all run
on this identity, usually under Kabsch's name, and the notebook builds it
from scratch — including the determinant correction that keeps a
reflection from masquerading as a rotation.

> **How to read a check.** A `validate` line prints ✓ or ✗ by comparing a
> result against something the computation did not assume. A ✗ flags a
> mismatch to investigate, never a verdict on its own.

> **Scope.** The polar decomposition and its Newton iteration are Higham
> {cite}`higham2008functions` Chapter 8; Golub and Van Loan
> {cite}`golub2013` §6.4 treats Procrustes. The closed-form rotation is
> Schönemann's {cite}`schonemann1966`; the determinant correction is
> Kabsch's {cite}`kabsch1976`.

## Theory in brief

### The factorization

From the SVD $A = U\Sigma V^{\top}$ of a square $A$, regroup:

```{math}
:label: eq-pp-polar
A \;=\; \underbrace{(UV^{\top})}_{Q}\;\underbrace{(V\Sigma V^{\top})}_{H},
\qquad Q^{\top}Q = I,\quad H = H^{\top}\succeq 0 .
```

$Q$ is orthogonal because it is a product of orthogonal factors; $H$ is
symmetric positive semidefinite because its eigenvalues are the singular
values. For invertible $A$ the pair is unique, and $H$ is *the* symmetric
positive semidefinite square root

```{math}
:label: eq-pp-sqrt
H \;=\; (A^{\top}A)^{1/2},
```

since $H^2 = V\Sigma^2V^{\top} = A^{\top}A$ and a psd matrix has exactly
one psd square root.

### Nearest orthogonal matrix

Among all orthogonal $W$,

```{math}
:label: eq-pp-nearest
Q \;=\; \operatorname*{argmin}_{W^{\top}W = I}\; \lVert A - W\rVert_F .
```

The proof is a trace computation the exercises verify numerically:
$\lVert A - W\rVert_F^2 = \lVert A\rVert_F^2 + n -
2\operatorname{tr}(W^{\top}A)$, so minimising the distance means
maximising the trace, and

```{math}
:label: eq-pp-trace
\operatorname{tr}(W^{\top}A) = \operatorname{tr}(W^{\top}U\Sigma V^{\top})
= \operatorname{tr}\bigl((V^{\top}W^{\top}U)\Sigma\bigr)
\;\le\; \sum_i \sigma_i ,
```

with equality exactly when $V^{\top}W^{\top}U = I$, i.e. $W = UV^{\top} =
Q$: an orthogonal matrix has entries at most 1, so the trace against a
diagonal $\Sigma$ is largest when the diagonal entries are all 1.

### Newton's iteration

The polar factor is also the fixed point of the deceptively simple

```{math}
:label: eq-pp-newton
Q_0 = A, \qquad Q_{k+1} = \tfrac12\bigl(Q_k + Q_k^{-\top}\bigr),
```

the matrix version of Heron's $x \leftarrow \tfrac12(x + 1/x)$ for
$\sqrt{1}$, applied to the singular values: each $\sigma$ flows to 1 at a
quadratic rate while the singular *vectors* never move, so the iterates
converge to $UV^{\top}$ with the error squaring at every step
{cite}`higham2008functions`.

### Procrustes, and the Kabsch correction

Stack $n$ points as the rows of $X, Y \in \mathbb{R}^{n\times2}$ and seek
the orthogonal $R$ making $\hat{\mathbf{y}}_i = R\mathbf{x}_i$ fit best:

```{math}
:label: eq-pp-procrustes
\hat R \;=\; \operatorname*{argmin}_{R^{\top}R = I}
\lVert XR^{\top} - Y\rVert_F \;=\; VU^{\top},
\qquad X^{\top}Y = U\Sigma V^{\top},
```

by the same trace argument as {eq}`eq-pp-trace` — Procrustes *is* the
nearest-orthogonal-matrix problem for the correlation matrix $X^{\top}Y$
{cite}`schonemann1966`. The unconstrained optimum may be a reflection
($\det \hat R = -1$); when the application demands a proper rotation,

```{math}
:label: eq-pp-kabsch
\hat R_{\mathrm{rot}} \;=\; V\,\operatorname{diag}(1, \det(VU^{\top}))\,U^{\top}
```

flips the least-important direction and is optimal among $\det = +1$
matrices {cite}`kabsch1976`.

### The left-handed twin

Regrouping the SVD the other way gives $A = H'Q$ with $H' = U\Sigma
U^{\top}$ — stretch *after* rotating:

```{math}
:label: eq-pp-left
A \;=\; QH \;=\; H'Q, \qquad H' = U\Sigma U^{\top} = AQ^{\top}.
```

The rotation is the *same* $Q$ in both; the stretches differ unless $A$ is
normal, and the gap $\lVert H - H'\rVert$ is a meter for how far $A$ is
from normality — the theme of
[§3.4](../03-eigenvalues/hermitian-unitary-normal.ipynb), met again from
the SVD side.

---
## Setup

Data only: the worked $3\times3$, the display $2\times2$ for the geometry
panels, and the Procrustes scene (a seeded spiral of 40 points, the
planted rotation, and its noisy image). Every method — the polar
factorization, Newton's iteration, the Procrustes solvers — is built in
the exercises, where it is the lesson.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import sqrtm

from ecp import validate
from ecp import linalg as la
from ecp.style import use_style

use_style()
rng = np.random.default_rng(0)  # every random array below comes from this seed

EPS = np.finfo(float).eps

# data: the worked 3x3 — invertible (det 6.25), condition number 4.25, and
# decisively non-normal, so Exercise 6's two stretches differ by something
# visible.
A_WORK = np.array([[2.0, 1.0, 0.0],
                   [0.5, 3.0, 1.0],
                   [0.0, 1.0, 1.5]])

# data: the display 2x2 for the geometry panels — an upper-triangular
# shear-and-stretch whose hidden rotation is not obvious by eye until the
# polar factor extracts it.
A_DISP = np.array([[1.4, 0.8],
                   [0.0, 0.9]])

# data: the Procrustes scene — N_PTS points on a seeded open spiral, rotated
# by THETA_TRUE and jittered at NOISE_P per coordinate. Points are rows.
N_PTS = 40
THETA_TRUE = 0.7
NOISE_P = 0.02
t_par = np.linspace(0.25, 2.4 * np.pi, N_PTS)
X_CLOUD = np.column_stack([0.08 * t_par * np.cos(t_par),
                           0.08 * t_par * np.sin(t_par)])
R_TRUE = np.array([[np.cos(THETA_TRUE), -np.sin(THETA_TRUE)],
                   [np.sin(THETA_TRUE),  np.cos(THETA_TRUE)]])
Y_CLOUD = X_CLOUD @ R_TRUE.T + NOISE_P * rng.standard_normal((N_PTS, 2))

## Exercise 1: One rotation, one stretch

{eq}`eq-pp-polar` says the SVD's three factors regroup into a rotation
and a symmetric stretch, and {eq}`eq-pp-sqrt` says the stretch is the
unique psd square root of $A^{\top}A$. This exercise builds the
factorization and checks both claims against routes that share no code.

**Part a)** Write `polar(A)` returning `(Q, H)` from one call to
`np.linalg.svd(A)`: $Q = UV^{\top}$ and $H = V\Sigma V^{\top}$ (as
`Vt.T @ (s[:, None] * Vt)`). On the $3\times3$ `A_WORK` of the Setup,
gate $\lVert A - QH\rVert_{\max} < 50\,\varepsilon\,\lVert A\rVert_2$,
$\lVert Q^{\top}Q - I\rVert_{\max} < 50\,\varepsilon$, and $H$ symmetric
to $50\,\varepsilon\,\lVert A\rVert_2$ with smallest eigenvalue above
$-50\,\varepsilon\,\lVert A\rVert_2$ (one-sided: its exact eigenvalues
are the singular values $3.77, 1.87, 0.89$, all positive).

**Write this one yourself** — the factorization is the notebook, and
every exercise below runs on the `polar` you build here.

**Part b)** Gate {eq}`eq-pp-sqrt` against an independent route:
$\lVert H^2 - A^{\top}A\rVert_{\max} < 100\,\varepsilon\,\lVert
A\rVert_2^2$, and $H$ against `scipy.linalg.sqrtm(A.T @ A)` — a Schur
method sharing no code with the SVD — to $10^{-11}$ relative in the
Frobenius norm.

**Part c)** See the regrouping on the display matrix `A_DISP`
($2\times2$): draw the unit circle's image under $H_D$ alone (a pure
stretch along the right singular directions, no rotation) beside its
image under the full $A_D$ — the same ellipse, turned by the polar
angle $\theta_Q = \operatorname{atan2}(Q_{21}, Q_{11}) = -19.2°$.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.below(
    recon_gap, 50 * EPS * norm_A,
    "A = QH reconstructs the matrix (Eq. 1)",
    "one SVD, regrouped — the rotation-times-stretch reading of 4.1's "
    "rotate-stretch-rotate",
)
validate.below(
    orth_gap, 50 * EPS,
    "Q is orthogonal to rounding",
    "a product of exactly orthogonal factors, spoiled only by arithmetic",
)
validate.check(
    sym_gap < 50 * EPS * norm_A and min_eig_H > -50 * EPS * norm_A,
    "and H is symmetric positive semidefinite (one-sided)",
    f"symmetry gap {sym_gap:.1e}; smallest eigenvalue {min_eig_H:.4f}, "
    "which is sigma_3 = 0.887 — the eigenvalues of H are the singular "
    "values of A",
)
validate.below(
    sq_gap, 100 * EPS * norm_A**2,
    "H squares back to A'A (Eq. 2)",
    "the defining property of the square root, checked as arithmetic",
)
validate.below(
    sqrtm_rel, 1e-11,
    "and the SVD's H matches the Schur-method square root",
    "np.linalg.svd against scipy.linalg.sqrtm: two factorizations sharing "
    "no code, one psd square root — uniqueness, measured",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2: The nearest orthogonal matrix

{eq}`eq-pp-nearest` claims optimality over *every* orthogonal matrix — a
claim quantified over a continuum, which sampling can refute but never
prove. As with Eckart–Young in [§4.2](low-rank-eckart-young.ipynb), we
do both things honesty allows: race a large random field against $Q$,
and verify the exact trace identity the proof runs on.

**Part a)** Draw $10^{4}$ random orthogonal challengers as `np.linalg.qr`
factors of seeded $3\times3$ Gaussians (a draw that lands on both
determinant signs), and gate that **none** comes closer to `A_WORK` in
the Frobenius norm than $Q$ does: $\min_W \lVert A - W\rVert_F >
\lVert A - Q\rVert_F = 2.906$. Report the best challenger's margin.

**Part b)** Gate the mechanism {eq}`eq-pp-trace`:
$\operatorname{tr}(Q^{\top}A) = \sum_i\sigma_i = 6.527$ to $10^{-12}$
relative, and every challenger's trace strictly below it — the distance
contest and the trace contest are the same contest, by the algebra in
the theory section.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    best_d > d_star,
    "no orthogonal challenger in 10^4 beats the polar factor (Eq. 3)",
    f"best {best_d:.3f} against {d_star:.3f} — sampling cannot prove a "
    "continuum claim, but it can lose to one, and it did, "
    f"{best_d / d_star:.2f}x over",
)
validate.check(
    abs(tr_star - sig_sum) < 1e-12 * sig_sum and worst_tr < sig_sum,
    "and the trace identity behind the proof holds exactly (Eq. 4)",
    f"tr(Q'A) = {tr_star:.6f} attains sum sigma while the best challenger "
    f"reaches {worst_tr:.4f}: minimising distance IS maximising trace",
)

## Exercise 3: Newton's iteration, and its quadratic engine

{eq}`eq-pp-newton` finds $Q$ without an SVD — the workhorse in codes
that need only the orthogonal factor. Its convergence is *quadratic*:
each singular value of the iterate obeys Heron's scalar recurrence
toward 1, so the error squares at every step.

**Part a)** Write `newton_polar(A, iters)` returning the list of
iterates $Q_0 = A, Q_1, \dots$ of {eq}`eq-pp-newton`, each step one
`np.linalg.inv` and a transpose. Run 9 iterations on `A_WORK`, measure
$e_k = \lVert Q_k - Q\rVert_F$ against Exercise 1's SVD factor, and
gate: $e_k$ first drops below $10^{-12}$ by iteration $k \le 8$
(measured: 6), and the final error sits below $10^{-13}$.

**Part b)** Gate the engine: for every consecutive pair with
$e_k \in (10^{-7}, 1)$ — the clean band above the rounding floor —
the ratio $e_{k+1}/e_k^2$ stays below $2$ (the theory constant is
$\approx 1/(2\sigma_{\min}) = 0.56$; measured ratios climb from
$0.12$ to $0.50$). Linear convergence cannot fake this: squaring is
the signature.

**Part c)** Draw $e_k$ on a log axis: the doubling slope of quadratic
convergence, six iterations from $O(1)$ to the floor.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    0 < k_conv <= 8 and errs_n[-1] < 1e-13,
    "Newton reaches the polar factor without an SVD (Eq. 5)",
    f"below 1e-12 at iteration {k_conv} of 9, final error "
    f"{errs_n[-1]:.1e} against the SVD's Q — two routes, one rotation",
)
validate.check(
    bool(np.all(ratios_n < 2.0)),
    "and the error genuinely squares: quadratic convergence, gated",
    f"e_(k+1)/e_k^2 at most {ratios_n.max():.2f} in the clean band, against "
    "a theory constant near 1/(2 sigma_min) = 0.56 — checked only above "
    "the 1e-7 floor, where squaring has room to show (5.4's window rule)",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4: Procrustes: the best rotation is a polar factor

{eq}`eq-pp-procrustes` solves the alignment problem in closed form. The
Setup planted it: `Y_CLOUD` is `X_CLOUD` rotated by $\theta = 0.7$ and
jittered at $\sigma_N = 0.02$ per coordinate.

**Part a)** Write `procrustes_rotation(X, Y)` returning
$\hat R = VU^{\top}$ from `np.linalg.svd(X.T @ Y)`, so that
$\hat{\mathbf{y}}_i = \hat R\mathbf{x}_i$. Recover the planted rotation
and gate the angle $\hat\theta = \operatorname{atan2}(\hat R_{21},
\hat R_{11})$ within the noise-permitted band
$|\hat\theta - 0.7| < 5\sigma_N/\sqrt{\sum_i\lVert\mathbf{x}_i\rVert^2}
= 0.044$ rad (the estimator's standard error is
$\sigma_N/\sqrt{\sum_i r_i^2}$; five of them is a generous ceiling over
a seeded draw). Report the measured error.

**Part b)** The reflection trap, sprung deliberately: reflect the cloud
across the line at angle $0.3$ (the $\det = -1$ matrix
$F = \left[\begin{smallmatrix}\cos 0.6 & \sin 0.6\\ \sin 0.6 &
-\cos 0.6\end{smallmatrix}\right]$, noiseless). Gate that unconstrained
Procrustes returns exactly this reflection ($\lVert\hat R - F\rVert_{\max}
< 10^{-12}$, $|\det\hat R + 1| < 10^{-12}$) — the *true* optimum, and
still the wrong answer for any application that promised a rotation.

**Part c)** Apply the Kabsch correction {eq}`eq-pp-kabsch` and certify
it: $|\det \hat R_{\mathrm{rot}} - 1| < 10^{-12}$, and over a sweep of
$3600$ rotations $R(\theta)$ the alignment error
$\lVert XR(\theta)^{\top} - Y\rVert_F$ never undercuts
$\hat R_{\mathrm{rot}}$'s by more than $10^{-9}$ — optimality among
proper rotations, raced against the whole circle.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.below(
    abs(theta_hat - THETA_TRUE), band_theta,
    "Procrustes recovers the planted rotation inside the noise band (Eq. 6)",
    f"angle error {abs(theta_hat - THETA_TRUE):.2e} against the "
    "5-standard-error ceiling — one SVD, no iteration, no initial guess",
)
validate.check(
    refl_gap < 1e-12 and abs(det_unc + 1.0) < 1e-12,
    "on the reflected scene the unconstrained optimum IS the reflection",
    f"R_hat = F to {refl_gap:.0e} with det {det_unc:+.3f}: the mathematics "
    "answered the question as posed, which is exactly the trap",
)
validate.check(
    abs(det_rot - 1.0) < 1e-12 and sweep_best > err_rot - 1e-9,
    "and the Kabsch correction is optimal among proper rotations (Eq. 7)",
    f"det +1 restored; none of 3600 swept rotations beats "
    f"{err_rot:.4f} — the flipped smallest direction is the cheapest "
    "concession a rotation can make",
)

## Exercise 5: Full Procrustes: scale, shift, rotate

Real alignment problems come with a scale and an offset. The classical
reduction: centre both clouds, estimate the rotation from the centred
correlation as in {eq}`eq-pp-procrustes`, read the scale off the same
SVD as $\hat s = \sum_i\sigma_i / \lVert X_c\rVert_F^2$, and recover the
shift as $\hat{\mathbf{t}} = \bar{\mathbf{y}} - \hat s\hat R
\bar{\mathbf{x}}$.

**Part a)** Plant the full scene: $\mathbf{y}_i = 1.6\,R_{0.7}\,
\mathbf{x}_i + (0.5, -0.3) + \boldsymbol{\epsilon}_i$ with fresh seeded
noise at $\sigma_N = 0.02$. Estimate $(\hat R, \hat s,
\hat{\mathbf{t}})$ by the reduction above and gate
$|\hat s/1.6 - 1| < 0.02$ and
$\lVert\hat{\mathbf{t}} - (0.5, -0.3)\rVert < 0.05$ — generous bands
for a $\sigma_N = 0.02$ scene, stated in advance.

**Part b)** Gate the fit itself: the RMS per-point residual of the
aligned cloud stays below $2\sigma_N\sqrt{2} = 0.057$ (each point
carries noise in two coordinates; the factor 2 is headroom), and
report it beside the noise floor.

**Part c)** Draw the scene: the two clouds before alignment, and the
transformed $\hat s\hat R X + \hat{\mathbf{t}}$ laid over $Y$ after.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.below(
    abs(s_full / S_TRUE - 1.0), 0.02,
    "the scale comes off the same SVD, inside its stated band",
    f"s_hat/s = {s_full / S_TRUE:.4f}: the singular values of the centred "
    "correlation carry the size of the map, not just its direction",
)
validate.below(
    float(np.linalg.norm(t_full - T_TRUE)), 0.05,
    "the shift follows from the two centroids, inside its stated band",
    f"||t_hat - t|| = {np.linalg.norm(t_full - T_TRUE):.4f} — centring "
    "first is what decouples the translation from the rotation",
)
validate.below(
    rms_fit, 2 * NOISE_P * np.sqrt(2),
    "and the aligned cloud sits at the noise floor",
    f"RMS {rms_fit:.4f} against the 2x-headroom ceiling "
    f"{2 * NOISE_P * np.sqrt(2):.4f}: nothing left to align away but "
    "the jitter itself",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6: Two stretches, one rotation — a normality meter

{eq}`eq-pp-left` regroups the same SVD the other way: stretch first or
rotate first, the rotation never changes, but the two stretches agree
only for normal matrices. That makes $\lVert H - H'\rVert$ a meter —
[§3.4](../03-eigenvalues/hermitian-unitary-normal.ipynb)'s commutator
test, reappearing from the SVD side.

**Part a)** Build the left factor as $H' = AQ^{\top}$ from Exercise 1's
`polar` output and gate the twin factorization:
$\lVert A - H'Q\rVert_{\max} < 50\,\varepsilon\,\lVert A\rVert_2$, with
$H'$ symmetric to the same scale and psd one-sided — the same three
checks as Validation 1, on the other grouping.

**Part b)** Gate the meter both ways on the same $10^{-2}$-scaled
yardstick: for the non-normal `A_WORK`,
$\lVert H - H'\rVert_F / \lVert H\rVert_F > 0.02$ (measured: $0.068$)
and equivalently $\lVert QH - HQ\rVert_F > 0.02\,\lVert A\rVert_F$ —
the factors refuse to commute by a visible margin. For the normal
specimen $A_N = 1.7\,R(0.5)$ (a scaled rotation), both gaps collapse
below $50\,\varepsilon\,\lVert A_N\rVert_2$: its stretch is $1.7 I$,
which commutes with everything.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    left_recon < 50 * EPS * norm_A and left_sym < 50 * EPS * norm_A
    and left_min_eig > -50 * EPS * norm_A,
    "A = H'Q holds with H' symmetric psd — the left-handed twin (Eq. 8)",
    f"reconstruction {left_recon:.1e}; the rotation is the SAME Q, only "
    "the stretch moved to the other side",
)
validate.check(
    meter_work > 0.02 and comm_work > 0.02,
    "the two stretches disagree visibly for the non-normal worked matrix",
    f"||H - H'||/||H|| = {meter_work:.3f} and the commutator at "
    f"{comm_work:.3f} of ||A||: stretch-then-rotate and rotate-then-stretch "
    "are different physical operations here",
)
validate.check(
    meter_nrm < 50 * EPS * 1.7 and comm_nrm < 50 * EPS * 1.7,
    "and collapse to rounding for the normal specimen",
    "a scaled rotation stretches every direction alike (H = 1.7 I), and a "
    "multiple of the identity commutes with its rotation — normality is "
    "exactly the property 3.4 said it was",
)

---
## Notebook summary

**One rotation, one stretch, certified twice.** The SVD's regrouping
$A = QH$ reconstructed the worked $3\times3$ at $10^{-15}$ with $Q$
orthogonal to rounding and $H$'s eigenvalues equal to the singular
values $(3.77, 1.87, 0.89)$; $H$ matched the Schur-method
`scipy.linalg.sqrtm(A.T @ A)` to $4\times10^{-16}$ relative — two
factorizations sharing no code, agreeing on the unique psd square root.

**Nearest means nearest.** None of $10^{4}$ random orthogonal
challengers came closer than $\lVert A - Q\rVert_F = 2.906$ (the best
lost by $1.31\times$), and the mechanism held exactly:
$\operatorname{tr}(Q^{\top}A)$ attained $\sum\sigma_i = 6.527$ at
rounding while every challenger's trace fell short.

**Newton squares its way there.** Nine SVD-free iterations of
$\tfrac12(Q + Q^{-\top})$ hit the polar factor at $4\times10^{-16}$,
crossing $10^{-12}$ at iteration 6, with clean-band ratios
$e_{k+1}/e_k^2$ between $0.12$ and $0.50$ against the theory constant
$1/(2\sigma_{\min}) = 0.56$ — squaring, measured, not asserted.

**Procrustes is a polar factor in disguise.** One SVD of $X^{\top}Y$
recovered the planted $\theta = 0.7$ to $9\times10^{-3}$ (band
$0.044$); the reflected scene returned its reflection *exactly*
($\det = -1$, the true optimum and the wrong answer), and the Kabsch
correction's proper rotation went unbeaten by all $3600$ swept
rotations. With scale and shift planted too, $(\hat s, \hat{\mathbf{t}},
\hat R)$ landed inside their stated bands and the aligned RMS sat at
the noise floor.

**The two stretches meter normality.** $A = QH = H'Q$ with the same
$Q$; the worked matrix's stretches disagreed by $6.8\%$ (its commutator
likewise) while the scaled rotation's collapsed to rounding —
[§3.4](../03-eigenvalues/hermitian-unitary-normal.ipynb)'s verdict,
reproduced by regrouping an SVD.

**Methods introduced.** `polar` via `np.linalg.svd`, the psd square
root {eq}`eq-pp-sqrt` against `scipy.linalg.sqrtm`, the
nearest-orthogonal certification race, `newton_polar` and its
clean-band quadratic gate, `procrustes_rotation` with the Kabsch
determinant correction and a sweep certificate, the centred
scale-and-shift reduction, and the $\lVert H - H'\rVert$ normality
meter.

## Outlook

- **Weighted Procrustes, and spacecraft.** Give each point pair its own
  confidence and the problem becomes Wahba's — attitude determination
  from star sightings, solved hourly in orbit by exactly this SVD with
  weights folded into the correlation matrix.
- **Orthogonalization by polar, not QR.** A rotation matrix drifting
  under repeated floating-point composition is best repaired by its
  polar factor — the *nearest* rotation, by Exercise 2 — where
  Gram–Schmidt would privilege the first column. Graphics and robotics
  codes re-orthogonalize this way for exactly that reason.
- **Scaled Newton and the matrix sign function.** Exercise 3's
  iteration converges slowly when $\kappa$ is large; Higham's scaling
  $Q_k \leftarrow \mu_k Q_k$ restores speed, and the same recurrence
  computes the matrix sign function at the heart of spectral
  divide-and-conquer eigensolvers
  ([§5.2](../05-numerical/eigenvalue-algorithms.ipynb) met the
  workhorses it competes with).
- **Shape statistics.** Quotienting clouds by rotation, scale and shift
  — exactly Exercise 5's estimates — is the starting point of Kendall's
  shape spaces, where a "mean shape" is a Procrustes average and
  [§4.3](pca-covariance-whitening.ipynb)'s PCA runs on the aligned
  residuals.

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()